## Depth Anything V2 — Inference & Testing Pipeline 

End-to-end pipeline for running **Depth Anything V2** (ViT-B) on a Waymo driving segment, 
converting raw PNG outputs to calibrated NumPy depth arrays, and aligning predictions against ground-truth depth.

---

**Pipeline Stages**
1. Configure paths
2. Verify inputs
3. Download model checkpoint
4. Load & validate data
5. Sanity-check model files
6. Run inference (subprocess)
7. Verify PNG outputs
8. Convert PNGs → normalised `.npy` arrays
9. Stack arrays into a single prediction tensor
10. Align predictions with ground truth
11. Apply calibration formula
12. Save final outputs

## Configure Paths

Define all repo, segment, and output directories. Creates `raw_png/`, `raw_npy/`, and `plots/` subdirectories if they don't exist.

In [ ]:
from pathlib import Path
import os

REPO_DIR = Path(r"/home/jiale/Documents/eco_cars")
DA_V2_DIR = REPO_DIR / "Depth-Anything-V2"

## Verify Input Directories

Sanity-check that the images directory and ground-truth stack files are present before proceeding.

In [3]:
print("IMAGES_DIR exists:", IMAGES_DIR.exists())
print("GT_STACK_PATH exists:", GT_STACK_PATH.exists())
print("GT_STEMS_PATH exists:", GT_STEMS_PATH.exists())

if IMAGES_DIR.exists():
    image_files = list(IMAGES_DIR.glob("*"))
    print("Number of files in images/:", len(image_files))
    print("First 5 image files:", [p.name for p in image_files[:5]])

IMAGES_DIR exists: True
GT_STACK_PATH exists: True
GT_STEMS_PATH exists: True
Number of files in images/: 197
First 5 image files: ['frame_00000.png', 'frame_00001.png', 'frame_00002.png', 'frame_00003.png', 'frame_00004.png']


## Download Model Checkpoint

Downloads the **ViT-B** checkpoint from HuggingFace if it isn't already cached locally.

In [3]:
from pathlib import Path
import urllib.request

CHECKPOINTS_DIR = DA_V2_DIR / "checkpoints"
CHECKPOINTS_DIR.mkdir(parents=True, exist_ok=True)

CKPT_PATH = CHECKPOINTS_DIR / "depth_anything_v2_vits.pth"
# CKPT_URL = "https://huggingface.co/depth-anything/Depth-Anything-V2-Base/resolve/main/depth_anything_v2_vitl.pth"
# CKPT_URL = "https://huggingface.co/depth-anything/Depth-Anything-V2-Large/resolve/main/depth_anything_v2_vitl.pth"
if not CKPT_PATH.exists():
    print("Downloading checkpoint...")
    urllib.request.urlretrieve(CKPT_URL, CKPT_PATH)
    print("Downloaded to:", CKPT_PATH)
else:
    print("Checkpoint already exists:", CKPT_PATH)

Checkpoint already exists: /home/jiale/Documents/eco_cars/Depth-Anything-V2/checkpoints/depth_anything_v2_vits.pth


## Load Images & Ground-Truth Depth

Globs input images and loads the ground-truth depth stack (`.npy`) along with its stem index. Asserts shapes are consistent.

In [5]:
import numpy as np
from pathlib import Path

image_files = sorted([p for p in IMAGES_DIR.iterdir() if p.suffix.lower() in [".png", ".jpg", ".jpeg"]])

assert len(image_files) > 0, f"No images found in {IMAGES_DIR}"
assert GT_STACK_PATH.exists(), f"Missing {GT_STACK_PATH}"
assert GT_STEMS_PATH.exists(), f"Missing {GT_STEMS_PATH}"

gt_stack = np.load(GT_STACK_PATH, allow_pickle=True)
gt_stems = np.load(GT_STEMS_PATH, allow_pickle=True)

print("num images:", len(image_files))
print("gt_stack shape:", gt_stack.shape)
print("gt_stems shape:", gt_stems.shape)
print("first 10 image stems:", [p.stem for p in image_files[:10]])
print("first 10 gt stems:", gt_stems[:10])

num images: 197
gt_stack shape: (197, 1280, 1920)
gt_stems shape: (197,)
first 10 image stems: ['frame_00000', 'frame_00001', 'frame_00002', 'frame_00003', 'frame_00004', 'frame_00005', 'frame_00006', 'frame_00007', 'frame_00008', 'frame_00009']
first 10 gt stems: ['frame_00000' 'frame_00001' 'frame_00002' 'frame_00003' 'frame_00004'
 'frame_00005' 'frame_00006' 'frame_00007' 'frame_00008' 'frame_00009']


## Sanity-Check Model Files

Verifies that `run.py` and the checkpoints directory exist inside the Depth-Anything-V2 repo before launching inference.

In [2]:
from pathlib import Path

RUN_PY = DA_V2_DIR / "run.py"
ckpt_dir = DA_V2_DIR / "checkpoints"

print("DA_V2_DIR exists:", DA_V2_DIR.exists())
print("run.py exists:", RUN_PY.exists())
print("checkpoints dir exists:", ckpt_dir.exists())

if ckpt_dir.exists():
    print("checkpoint files:")
    for p in ckpt_dir.iterdir():
        print(" -", p.name)

DA_V2_DIR exists: True
run.py exists: True
checkpoints dir exists: True
checkpoint files:
 - depth_anything_v2_vitl.pth


## Run Depth Anything V2 Inference

Launches `run.py` via subprocess with `--encoder vitb`, `--pred-only`, and `--grayscale` flags. Predicted depth PNGs are written to `raw_png/`.

In [ ]:
import subprocess
import sys
from pathlib import Path
import time

run_py = DA_V2_DIR / "run.py"
assert run_py.exists(), f"Cannot find run.py at {run_py}"

def run_inference(IMAGES_DIR, RAW_PNG_DIR):
    before = time.time()

    cmd = [
        sys.executable, str(run_py),
        "--encoder", "vitl",
        "--img-path", str(IMAGES_DIR),
        "--outdir", str(RAW_PNG_DIR),
        "--pred-only"
    ]

    print("Running inference...")
    print(" ".join(cmd))


    result = subprocess.run(
        cmd,
        cwd=str(DA_V2_DIR),
        capture_output=True,
        text=True
    )

    # Caculating the inference speed and frame rate
    after = time.time()
    frame_rate = len(list(IMAGES_DIR.glob("*"))) / (after - before)

    print(f"Inference Speed per t_record: {after-before}")
    print(f"Frame rate: {frame_rate}")
    print("return code:", result.returncode)
    print("STDOUT:\n", result.stdout)
    print("STDERR:\n", result.stderr)

    if result.returncode != 0:
        raise RuntimeError("Inference failed.")
    else:
        print("Inference complete.")

## Verify PNG Outputs

Lists the predicted PNG files to confirm inference produced the expected number of outputs.

In [8]:
pred_pngs = sorted(RAW_PNG_DIR.glob("*.png"))
print("num predicted pngs:", len(pred_pngs))
print("first 10 predicted files:", [p.name for p in pred_pngs[:10]])

num predicted pngs: 197
first 10 predicted files: ['frame_00000.png', 'frame_00001.png', 'frame_00002.png', 'frame_00003.png', 'frame_00004.png', 'frame_00005.png', 'frame_00006.png', 'frame_00007.png', 'frame_00008.png', 'frame_00009.png']


## Convert PNGs → Normalised NumPy Arrays

Reads each predicted grayscale PNG with OpenCV, normalises pixel values to **[0, 1]**, and saves individual `.npy` files to `raw_npy/`.

In [ ]:
import cv2
import numpy as np
from pathlib import Path
import shutil

def normalised_numpy_depths(self, RAW_PNG_DIR, RAW_NPY_DIR):
    depth_png_files = sorted([p for p in RAW_PNG_DIR.iterdir() if p.suffix.lower() == ".png"])
    assert len(depth_png_files) > 0, f"No depth PNG files found in {RAW_PNG_DIR}"

    print(f"Found {len(depth_png_files)} predicted depth PNGs")
    for i, path in enumerate(RAW_PNG_DIR):
        img = cv2.imread(path, cv2.IMREAD_GRAYSCALE).astype(np.float32)

        if img.max() > img.min():
            img_normalized = (img - img.min()) / (img.max() - img.min())
        else:
            img_normalized = img.copy()

        img_normalized = img / 255.0
        # flip: now bright = far, dark = close
        img_inverted = 1.0 - img_normalized   
        # np.save(RAW_NPY_DIR / f'{stem}_depth.npy', img_inverted)
        stem = Path(path).stem
        np.save(RAW_NPY_DIR / f'{stem}_depth.npy', img_inverted)
        if i % 20 == 0:
            print(f'  [{i+1}/{len(RAW_NPY_DIR)}] {stem}  shape={img_normalized.shape}  range=[{img_normalized.min():.2f}, {img_normalized.max():.2f}]')

    print('Done!')
    # for path in depth_png_files:
    #     img = cv2.imread(str(path), cv2.IMREAD_UNCHANGED)

    #     if img is None:
    #         print(f"Warning: failed to read {path}")
    #         continue

    #     if img.ndim == 3:
    #         img = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)

    #     img = img.astype(np.float32)

    #     if img.max() > img.min():
    #         img_normalized = (img - img.min()) / (img.max() - img.min())
    #     else:
    #         img_normalized = img.copy()

    #     stem = path.stem
    #     np.save(RAW_NPY_DIR / f"{stem}_depth.npy", img_normalized)

    print("Saved individual depth arrays to:", RAW_NPY_DIR)
    shutil.make_archive(RAW_PNG_DIR, "zip", RAW_PNG_DIR)

    shutil.rmtree(RAW_PNG_DIR)

## Stack Individual Arrays into Prediction Tensor

Loads all `*_depth.npy` files, stacks them along axis 0, and saves a single `depthanythingv2_depths.npy` tensor together with a `stems.npy` index.

In [5]:
import numpy as np
from pathlib import Path
import shutil

def stack_numpy(self, RAW_NPY_DIR, OUTPUT_DIR):
    pred_npy_files = sorted(RAW_NPY_DIR.glob("*_depth.npy"))
    assert len(pred_npy_files) > 0, f"No prediction npy files found in {RAW_NPY_DIR}"

    pred_stack = []
    pred_stems = []

    for f in pred_npy_files:
        arr = np.load(f)
        pred_stack.append(arr)
        pred_stems.append(f.stem.replace("_depth", ""))

    pred_stack = np.stack(pred_stack, axis=0)
    pred_stems = np.array(pred_stems)

    PRED_STACK_PATH = OUTPUT_DIR / "depthanythingv2_depths.npy"
    PRED_STEMS_PATH = OUTPUT_DIR / "depthanythingv2_stems.npy"

    np.save(PRED_STACK_PATH, pred_stack)
    np.save(PRED_STEMS_PATH, pred_stems)

    print("pred_stack shape:", pred_stack.shape)
    print("Saved:", PRED_STACK_PATH)
    print("Saved:", PRED_STEMS_PATH)

    shutil.make_archive(PRED_STACK_PATH, "zip", PRED_STACK_PATH)
    shutil.make_archive(PRED_STEMS_PATH, "zip", PRED_STEMS_PATH)

    shutil.rmtree(PRED_STACK_PATH)
    shutil.rmtree(PRED_STEMS_PATH)

## Running inference on multiple record files

In [ ]:
import io
import os
import sys
import shutil
import subprocess
import zipfile
import queue
import threading
import time
from pathlib import Path
from concurrent.futures import ThreadPoolExecutor

import cv2
import numpy as np


# ================= CONFIG =================

INPUT_ROOT  = Path("/home/jiale/Documents/eco_cars/data/output_images")
OUTPUT_ROOT = Path("/home/jiale/Documents/eco_cars/data/depth_anything")
TEMP_ROOT   = OUTPUT_ROOT / "temp"

DA_V2_DIR = Path("/home/jiale/Documents/eco_cars/Depth-Anything-V2")
RUN_PY    = DA_V2_DIR / "run.py"

NUM_UNZIP_WORKERS = 2
NUM_POST_WORKERS  = 4   # bump to keep up with 4fps GPU
NUM_GPU_WORKERS   = 1

MAX_QUEUE_SIZE = 4

OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
TEMP_ROOT.mkdir(parents=True, exist_ok=True)

# Top-level output buckets — one zip per segment lands here
IMAGES_ZIP_DIR     = OUTPUT_ROOT / "images"
DEPTHS_ZIP_DIR     = OUTPUT_ROOT / "depth"
ALL_DEPTHS_ZIP_DIR = OUTPUT_ROOT / "all_depths"

for d in (IMAGES_ZIP_DIR, DEPTHS_ZIP_DIR, ALL_DEPTHS_ZIP_DIR):
    d.mkdir(parents=True, exist_ok=True)

# QUEUES
unzip_queue     = queue.Queue(maxsize=MAX_QUEUE_SIZE)
inference_queue = queue.Queue(maxsize=MAX_QUEUE_SIZE)
post_queue      = queue.Queue(maxsize=MAX_QUEUE_SIZE)

print_lock = threading.Lock()

def safe_print(*a):
    with print_lock:
        print(*a, flush=True)


# HELPERS

def png_to_norm_float32(path: Path):
    """Read a grayscale depth PNG, normalise to [0,1] float32. Returns None on failure."""
    img = cv2.imread(str(path), cv2.IMREAD_UNCHANGED)
    if img is None:
        return None
    if img.ndim == 3:
        img = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    img = img.astype(np.float32)
    lo, hi = img.min(), img.max()
    if hi > lo:
        img = (img - lo) / (hi - lo)
    return img


def arr_to_npy_bytes(arr: np.ndarray) -> bytes:
    """Serialise a numpy array to raw .npy bytes (in memory, no temp file)."""
    buf = io.BytesIO()
    np.save(buf, arr)
    return buf.getvalue()


def arrays_to_npz_bytes(**kwargs) -> bytes:
    """Serialise one or more arrays to compressed .npz bytes (in memory)."""
    buf = io.BytesIO()
    np.savez_compressed(buf, **kwargs)
    return buf.getvalue()


# UNZIP the individual files
def unzip_worker():
    while True:
        zip_path = unzip_queue.get()
        if zip_path is None:
            break
        try:
            name        = zip_path.stem
            extract_dir = TEMP_ROOT / name
            extract_dir.mkdir(parents=True, exist_ok=True)

            # Extract member-by-member — avoids loading the whole archive at once
            with zipfile.ZipFile(zip_path, "r") as z:
                for member in z.infolist():
                    z.extract(member, extract_dir)

            safe_print(f"[UNZIP] {name}")
            inference_queue.put((name, extract_dir))

        except Exception as e:
            safe_print(f"[UNZIP ERROR] {zip_path.name}: {e}")
        finally:
            unzip_queue.task_done()


# RUN gpu inference on each individual workers
def inference_worker():
    while True:
        item = inference_queue.get()
        if item is None:
            break

        name, extract_dir = item
        png_dir = TEMP_ROOT / f"{name}_png"
        png_dir.mkdir(parents=True, exist_ok=True)

        image_files = sorted(
            p for p in extract_dir.iterdir()
            if p.suffix.lower() in (".png", ".jpg", ".jpeg")
        )
        n_frames = len(image_files)

        try:
            cmd = [
                sys.executable, str(RUN_PY),
                "--encoder",  "vitl",
                "--img-path", str(extract_dir),
                "--outdir",   str(png_dir),
                "--pred-only",
                "--grayscale",
            ]
            t_start  = time.perf_counter()
            result = subprocess.run(
                cmd, cwd=str(DA_V2_DIR), capture_output=True, text=True, timeout=600
            )

            elapsed    = time.perf_counter() - t_start
            fps        = n_frames / elapsed if elapsed > 0 else float("nan")

            # always print timing, even on failure
            safe_print(
                f"[GPU] {name}  |  "
                f"frames={n_frames}  time={elapsed:.2f}s  fps={fps:.2f}"
            )
            if result.stdout.strip():
                safe_print(f"[GPU stdout]\n{result.stdout.strip()}")
            if result.stderr.strip():
                safe_print(f"[GPU stderr]\n{result.stderr.strip()}")

            if result.returncode != 0:
                raise RuntimeError(f"run.py exited {result.returncode}")

            post_queue.put((name, extract_dir, png_dir, image_files))

        except Exception as e:
            safe_print(f"[GPU ERROR] {name}: {e}")
            shutil.rmtree(png_dir,     ignore_errors=True)
            shutil.rmtree(extract_dir, ignore_errors=True)
        finally:
            inference_queue.task_done()


def post_worker():
    while True:
        item = post_queue.get()
        if item is None:
            break

        name, extract_dir, png_dir, image_files = item

        try:
            png_files = sorted(png_dir.glob("*.png"))
            assert png_files, f"No depth PNGs in {png_dir}"

            # OUTPUT STRUCTURE:
            #   images/     <segment>.zip  → original input frames
            #   depth/      <segment>.zip  → per-frame <stem>_depth.npy files
            #   all_depths/ <segment>.zip  → single <segment>_all_depths.npz

            # Stream each source frame straight into the zip — zero copies in RAM.
            images_zip = IMAGES_ZIP_DIR / f"{name}.zip"
            with zipfile.ZipFile(images_zip, "w",
                                 compression=zipfile.ZIP_DEFLATED,
                                 compresslevel=1, allowZip64=True) as zf:
                for src in image_files:
                    zf.write(src, arcname=src.name)

            safe_print(f"[POST] {name}  images.zip ✓")

            # Convert one PNG at a time → serialise to bytes → write into zip.
            # The float32 array is deleted immediately after serialisation so
            # RAM stays at ~1 frame, not N frames.
            # We keep the serialised npy bytes around (they're much smaller)
            # to reuse when building all_depths without re-reading PNGs.
            depths_zip = DEPTHS_ZIP_DIR / f"{name}.zip"
            stems      = []
            npy_bufs   = []   # list[bytes] — reused for all_depths step

            with zipfile.ZipFile(depths_zip, "w",
                                 compression=zipfile.ZIP_DEFLATED,
                                 compresslevel=1, allowZip64=True) as zf:
                for p in png_files:
                    arr = png_to_norm_float32(p)
                    if arr is None:
                        safe_print(f"[POST WARN] skipping unreadable {p.name}")
                        continue

                    npy_bytes = arr_to_npy_bytes(arr)
                    del arr                          

                    stems.append(p.stem)
                    npy_bufs.append(npy_bytes)
                    zf.writestr(f"{p.stem}_depth.npy", npy_bytes)

            safe_print(f"[POST] {name}  depth.zip ✓  ({len(stems)} frames)")

            # Deserialise each npy buffer back to array, stack once, compress.
            # Peak RAM here = the stacked float32 tensor (N×H×W×4 bytes).
            # We immediately delete the stack after npz serialisation.
            frames    = [np.load(io.BytesIO(b)) for b in npy_bufs]
            del npy_bufs                             # free serialised buffers

            stack     = np.stack(frames, axis=0)    # (N, H, W) float32
            stems_arr = np.array(stems)
            del frames                               # free per-frame arrays

            npz_bytes = arrays_to_npz_bytes(depth_stack=stack, stems=stems_arr)
            del stack, stems_arr                     # free stack immediately

            all_depths_zip = ALL_DEPTHS_ZIP_DIR / f"{name}.zip"
            with zipfile.ZipFile(all_depths_zip, "w",
                                 compression=zipfile.ZIP_DEFLATED,
                                 compresslevel=1, allowZip64=True) as zf:
                zf.writestr(f"{name}_all_depths.npz", npz_bytes)
            del npz_bytes

            safe_print(f"[POST] {name}  all_depths.zip ✓")

            #Clean up temp dirs
            shutil.rmtree(png_dir,     ignore_errors=True)
            shutil.rmtree(extract_dir, ignore_errors=True)

            safe_print(f"[POST] {name}  DONE ✓")

        except Exception as e:
            safe_print(f"[POST ERROR] {name}: {e}")
            shutil.rmtree(png_dir,     ignore_errors=True)
            shutil.rmtree(extract_dir, ignore_errors=True)
        finally:
            post_queue.task_done()


# MAIN
def main():
    zip_files = sorted(INPUT_ROOT.glob("*.zip"))
    safe_print(f"Found {len(zip_files)} segment zip(s)")

    total_workers = NUM_UNZIP_WORKERS + NUM_GPU_WORKERS + NUM_POST_WORKERS

    with ThreadPoolExecutor(max_workers=total_workers) as executor:
        for _ in range(NUM_UNZIP_WORKERS):
            executor.submit(unzip_worker)
        for _ in range(NUM_GPU_WORKERS):
            executor.submit(inference_worker)
        for _ in range(NUM_POST_WORKERS):
            executor.submit(post_worker)

        for z in zip_files:
            unzip_queue.put(z)

        # Drain each stage before sending poison pills downstream
        unzip_queue.join()

        for _ in range(NUM_GPU_WORKERS):
            inference_queue.put(None)
        inference_queue.join()

        for _ in range(NUM_POST_WORKERS):
            post_queue.put(None)
        post_queue.join()

        for _ in range(NUM_UNZIP_WORKERS):
            unzip_queue.put(None)

    safe_print("ALL DONE")


if __name__ == "__main__":
    main()

Found 22 segment zip(s)
[UNZIP] segment-10161761842905385678_760_000_780_000_with_camera_labels
[UNZIP] segment-10534368980139017457_4480_000_4500_000_with_camera_labels
[UNZIP] segment-10649066155322078676_1660_000_1680_000_with_camera_labels
[UNZIP] segment-10998289306141768318_1280_000_1300_000_with_camera_labels
[UNZIP] segment-12892154548237137398_2820_000_2840_000_with_camera_labels
[UNZIP] segment-1376304843325714018_3420_000_3440_000_with_camera_labels
[UNZIP] segment-13790309965076620852_6520_000_6540_000_with_camera_labels
[GPU] segment-10161761842905385678_760_000_780_000_with_camera_labels  |  frames=198  time=118.28s  fps=1.67
[GPU stdout]
Progress 1/198: /home/jiale/Documents/eco_cars/data/depth_anything/temp/segment-10161761842905385678_760_000_780_000_with_camera_labels/frame_00177.png
Progress 2/198: /home/jiale/Documents/eco_cars/data/depth_anything/temp/segment-10161761842905385678_760_000_780_000_with_camera_labels/frame_00131.png
Progress 3/198: /home/jiale/Documen

In [ ]:
# ──────────────────────────────────────────────────────────────────────────────
# Running inference on multiple record files
#
# Pipeline:  unzip  →  GPU inference  →  normalise + stack  →  zip + clean up
# Each stage runs in its own thread(s) and communicates via bounded queues,
# so disk I/O overlaps with GPU compute instead of serialising.
# ──────────────────────────────────────────────────────────────────────────────
import os
import sys
import time
import queue
import shutil
import zipfile
import threading
import subprocess
from pathlib import Path
from concurrent.futures import ThreadPoolExecutor

import cv2
import numpy as np


# ─────────────────────────────── CONFIG ──────────────────────────────────────
INPUT_ROOT  = Path("/home/jiale/Documents/eco_cars/data/output_images")      # contains <segment>.zip files of raw frames
OUTPUT_ROOT = Path("/home/jiale/Documents/eco_cars/data/depth_anything")     # where final per-segment zips land
TEMP_ROOT   = OUTPUT_ROOT / "_temp"                                          # scratch area (auto-cleaned)

DA_V2_DIR   = Path("/home/jiale/Documents/eco_cars/Depth-Anything-V2")
RUN_PY      = DA_V2_DIR / "run.py"
assert RUN_PY.exists(), f"Cannot find run.py at {RUN_PY}"

# One GPU → one inference worker. Unzip and post can parallelise freely.
NUM_UNZIP_WORKERS = 2
NUM_GPU_WORKERS   = 1
NUM_POST_WORKERS  = 3

# Keep queues small so a slow stage applies backpressure instead of ballooning disk use.
MAX_QUEUE_SIZE = 3

# Final output layout: one zip per segment per output type.
IMAGES_ZIP_DIR     = OUTPUT_ROOT / "images"        # original frames, re-zipped
DEPTH_NPY_ZIP_DIR  = OUTPUT_ROOT / "depth_npy"     # per-frame *_depth.npy
DEPTH_STACK_DIR    = OUTPUT_ROOT / "depth_stack"   # single stacked .npy per segment

for d in (OUTPUT_ROOT, TEMP_ROOT, IMAGES_ZIP_DIR, DEPTH_NPY_ZIP_DIR, DEPTH_STACK_DIR):
    d.mkdir(parents=True, exist_ok=True)


# ─────────────────────── Cleaned-up core functions ───────────────────────────
# These are the same three helpers from earlier cells, with the bugs fixed:
#   • `self` parameter removed (they weren't methods)
#   • `normalised_numpy_depths`: iterate the file list, not the directory Path;
#     pass str() to cv2.imread; drop dead min/max branch.
#   • `stack_numpy`: use Path.unlink() on files, not shutil.rmtree() which is
#     for directories only; don't auto-archive inside the helper (the pipeline
#     handles zipping at the end).

def run_inference(images_dir: Path, raw_png_dir: Path) -> None:
    """Run Depth-Anything-V2 on every image in `images_dir`, writing PNGs to `raw_png_dir`."""
    before = time.time()
    cmd = [
        sys.executable, str(RUN_PY),
        "--encoder",  "vitl",
        "--img-path", str(images_dir),
        "--outdir",   str(raw_png_dir),
        "--pred-only",
        "--grayscale",
    ]
    result = subprocess.run(
        cmd, cwd=str(DA_V2_DIR),
        capture_output=True, text=True, timeout=1800,
    )
    elapsed    = time.time() - before
    n_frames   = sum(1 for _ in images_dir.iterdir())
    frame_rate = n_frames / elapsed if elapsed > 0 else float("nan")

    if result.returncode != 0:
        safe_print(f"[GPU stderr] {result.stderr.strip()}")
        raise RuntimeError(f"run.py exited {result.returncode}")

    safe_print(
        f"[GPU] {images_dir.name}  frames={n_frames}  "
        f"time={elapsed:.1f}s  fps={frame_rate:.2f}"
    )


def normalised_numpy_depths(raw_png_dir: Path, raw_npy_dir: Path) -> None:
    """Convert every depth PNG in `raw_png_dir` to a normalised+inverted .npy in `raw_npy_dir`.

    Normalisation uses /255.0 so values are comparable across frames (global scale),
    then inverted so bright=far, dark=close becomes dark=far, bright=close.
    """
    raw_npy_dir.mkdir(parents=True, exist_ok=True)
    depth_png_files = sorted(p for p in raw_png_dir.iterdir() if p.suffix.lower() == ".png")
    assert depth_png_files, f"No depth PNG files found in {raw_png_dir}"

    for path in depth_png_files:
        img = cv2.imread(str(path), cv2.IMREAD_GRAYSCALE)
        if img is None:
            safe_print(f"[POST WARN] unreadable: {path.name}")
            continue
        img_norm     = img.astype(np.float32) / 255.0
        img_inverted = 1.0 - img_norm
        np.save(raw_npy_dir / f"{path.stem}_depth.npy", img_inverted)


def stack_numpy(raw_npy_dir: Path, output_dir: Path, segment_name: str) -> tuple[Path, Path]:
    """Stack all per-frame .npy files into a single (N, H, W) array + stem list.

    Returns the two paths that were written.
    """
    pred_npy_files = sorted(raw_npy_dir.glob("*_depth.npy"))
    assert pred_npy_files, f"No *_depth.npy files in {raw_npy_dir}"

    frames = [np.load(f) for f in pred_npy_files]
    stems  = [f.stem.replace("_depth", "") for f in pred_npy_files]

    pred_stack = np.stack(frames, axis=0)
    pred_stems = np.array(stems)

    stack_path = output_dir / f"{segment_name}_depths.npy"
    stems_path = output_dir / f"{segment_name}_stems.npy"
    np.save(stack_path, pred_stack)
    np.save(stems_path, pred_stems)

    safe_print(f"[POST] {segment_name}  stack shape={pred_stack.shape}")
    return stack_path, stems_path


# ─────────────────────────────── PLUMBING ─────────────────────────────────────
unzip_queue     = queue.Queue(maxsize=MAX_QUEUE_SIZE)   # Path (input zip)
inference_queue = queue.Queue(maxsize=MAX_QUEUE_SIZE)   # (name, extract_dir)
post_queue      = queue.Queue(maxsize=MAX_QUEUE_SIZE)   # (name, extract_dir, png_dir)

print_lock = threading.Lock()
def safe_print(*a):
    with print_lock:
        print(*a, flush=True)


def _zip_dir(src_dir: Path, out_zip: Path) -> None:
    """Write every file under src_dir into out_zip, preserving relative names."""
    with zipfile.ZipFile(out_zip, "w",
                         compression=zipfile.ZIP_DEFLATED,
                         compresslevel=1, allowZip64=True) as zf:
        for f in sorted(src_dir.rglob("*")):
            if f.is_file():
                zf.write(f, arcname=f.relative_to(src_dir))


# ─────────────────────────────── WORKERS ──────────────────────────────────────
def unzip_worker():
    while True:
        zip_path = unzip_queue.get()
        try:
            if zip_path is None:
                return
            name        = zip_path.stem
            extract_dir = TEMP_ROOT / f"{name}_frames"
            if extract_dir.exists():
                shutil.rmtree(extract_dir)
            extract_dir.mkdir(parents=True)

            with zipfile.ZipFile(zip_path, "r") as z:
                z.extractall(extract_dir)

            # Flatten in case the zip has a single top-level folder.
            entries = list(extract_dir.iterdir())
            if len(entries) == 1 and entries[0].is_dir():
                inner = entries[0]
                for child in inner.iterdir():
                    child.rename(extract_dir / child.name)
                inner.rmdir()

            safe_print(f"[UNZIP] {name}")
            inference_queue.put((name, extract_dir))
        except Exception as e:
            safe_print(f"[UNZIP ERROR] {zip_path}: {e}")
        finally:
            unzip_queue.task_done()


def inference_worker():
    while True:
        item = inference_queue.get()
        try:
            if item is None:
                return
            name, extract_dir = item
            png_dir = TEMP_ROOT / f"{name}_png"
            if png_dir.exists():
                shutil.rmtree(png_dir)
            png_dir.mkdir(parents=True)

            try:
                run_inference(extract_dir, png_dir)
                post_queue.put((name, extract_dir, png_dir))
            except Exception as e:
                safe_print(f"[GPU ERROR] {name}: {e}")
                shutil.rmtree(png_dir,     ignore_errors=True)
                shutil.rmtree(extract_dir, ignore_errors=True)
        finally:
            inference_queue.task_done()


def post_worker():
    while True:
        item = post_queue.get()
        try:
            if item is None:
                return
            name, extract_dir, png_dir = item
            npy_dir = TEMP_ROOT / f"{name}_npy"
            if npy_dir.exists():
                shutil.rmtree(npy_dir)
            npy_dir.mkdir(parents=True)

            try:
                # 1. Re-zip the original input frames for archival.
                images_zip = IMAGES_ZIP_DIR / f"{name}.zip"
                _zip_dir(extract_dir, images_zip)
                safe_print(f"[POST] {name}  images.zip ✓")

                # 2. PNG → normalised inverted .npy (one file per frame).
                normalised_numpy_depths(png_dir, npy_dir)
                npy_zip = DEPTH_NPY_ZIP_DIR / f"{name}.zip"
                _zip_dir(npy_dir, npy_zip)
                safe_print(f"[POST] {name}  depth_npy.zip ✓")

                # 3. Stack into a single (N, H, W) tensor + stems.
                stack_numpy(npy_dir, DEPTH_STACK_DIR, name)
                safe_print(f"[POST] {name}  DONE ✓")
            except Exception as e:
                safe_print(f"[POST ERROR] {name}: {e}")
            finally:
                # Always clean temp dirs, success or failure.
                shutil.rmtree(extract_dir, ignore_errors=True)
                shutil.rmtree(png_dir,     ignore_errors=True)
                shutil.rmtree(npy_dir,     ignore_errors=True)
        finally:
            post_queue.task_done()


# ─────────────────────────────── MAIN ─────────────────────────────────────────
def main():
    zip_files = sorted(INPUT_ROOT.glob("*.zip"))
    safe_print(f"Found {len(zip_files)} segment zip(s) under {INPUT_ROOT}")
    if not zip_files:
        return

    total_workers = NUM_UNZIP_WORKERS + NUM_GPU_WORKERS + NUM_POST_WORKERS
    with ThreadPoolExecutor(max_workers=total_workers) as pool:
        for _ in range(NUM_UNZIP_WORKERS):  pool.submit(unzip_worker)
        for _ in range(NUM_GPU_WORKERS):    pool.submit(inference_worker)
        for _ in range(NUM_POST_WORKERS):   pool.submit(post_worker)

        # Feed the pipeline.
        for z in zip_files:
            unzip_queue.put(z)

        # Drain each stage, then send poison pills to the NEXT stage.
        # Order matters: finish stage N before telling stage N+1 it can stop.
        unzip_queue.join()
        for _ in range(NUM_UNZIP_WORKERS): unzip_queue.put(None)

        inference_queue.join()
        for _ in range(NUM_GPU_WORKERS):   inference_queue.put(None)

        post_queue.join()
        for _ in range(NUM_POST_WORKERS):  post_queue.put(None)

    # Final sweep of anything we missed.
    shutil.rmtree(TEMP_ROOT, ignore_errors=True)
    safe_print("ALL DONE ✓")


main()

Found 22 segment zip(s) under /home/jiale/Documents/eco_cars/data/output_images
[UNZIP] segment-10161761842905385678_760_000_780_000_with_camera_labels
[UNZIP] segment-10534368980139017457_4480_000_4500_000_with_camera_labels
[UNZIP] segment-10649066155322078676_1660_000_1680_000_with_camera_labels
[UNZIP] segment-10998289306141768318_1280_000_1300_000_with_camera_labels
[UNZIP] segment-12892154548237137398_2820_000_2840_000_with_camera_labels
[UNZIP] segment-1376304843325714018_3420_000_3440_000_with_camera_labels
[GPU] segment-10161761842905385678_760_000_780_000_with_camera_labels_frames  frames=198  time=64.4s  fps=3.07
[UNZIP] segment-13790309965076620852_6520_000_6540_000_with_camera_labels
[POST] segment-10161761842905385678_760_000_780_000_with_camera_labels  images.zip ✓
[POST] segment-10161761842905385678_760_000_780_000_with_camera_labels  depth_npy.zip ✓
[POST] segment-10161761842905385678_760_000_780_000_with_camera_labels  stack shape=(198, 1280, 1920)
[POST] segment-1016

## Align Predictions with Ground Truth

Matches prediction and GT arrays by stem name, filters to the common subset, and stacks both into aligned tensors ready for metric evaluation.

In [11]:
pred_stack = np.load(PRED_STACK_PATH, allow_pickle=True)
pred_stems = np.load(PRED_STEMS_PATH, allow_pickle=True)
gt_stack = np.load(GT_STACK_PATH, allow_pickle=True)
gt_stems = np.load(GT_STEMS_PATH, allow_pickle=True)

pred_dict = {stem: pred_stack[i] for i, stem in enumerate(pred_stems)}
gt_dict = {stem: gt_stack[i] for i, stem in enumerate(gt_stems)}

common_stems = sorted(set(pred_dict.keys()) & set(gt_dict.keys()))
assert len(common_stems) > 0, "No matching stems between prediction and ground truth"

pred_aligned = np.stack([pred_dict[s] for s in common_stems], axis=0)
gt_aligned = np.stack([gt_dict[s] for s in common_stems], axis=0)

print("Matched stems:", len(common_stems))
print("pred_aligned shape:", pred_aligned.shape)
print("gt_aligned shape:", gt_aligned.shape)
print("first 10 matched stems:", common_stems[:10])

Matched stems: 197
pred_aligned shape: (197, 1280, 1920)
gt_aligned shape: (197, 1280, 1920)
first 10 matched stems: ['frame_00000', 'frame_00001', 'frame_00002', 'frame_00003', 'frame_00004', 'frame_00005', 'frame_00006', 'frame_00007', 'frame_00008', 'frame_00009']


## Apply Calibration Formula

In [ ]:
## Calculating the calibration formula based out on the first 5 frames
import numpy as np
from scipy import stats
from scipy.optimize import curve_fit
import matplotlib.pyplot as plt
import os
print(os.getcwd())

# Removing the outliers using iqr method and finding the standard deviation in the curve
def noiseHandler(pred40, gt40): 
    # Creating a generic mask on top of the ground truth and relative data for filtering
    groundTruth = gt40.flatten()
    predictions = pred40.flatten()

    # Using iqr to remove the outliers for the graph taking 20 outliears 
    totalBins = 20
    numberOfBins = np.linspace(predictions.min(), predictions.max(), totalBins+1) 
    # Getting the index of each outlier
    numberOfBins_index = np.digitize(predictions, numberOfBins) - 1
    # Starting everything with 0 by default
    binsInlier = np.zeros(len(predictions), dtype=bool)    

    minimum_entries = 25
    for individualBins in range(totalBins):
        binsTotalEntries = (numberOfBins_index == individualBins)        
        if binsTotalEntries.sum() < minimum_entries:
            continue

        # Getting the individual entries in the area of the graph
        goundtruth_entry = groundTruth[binsTotalEntries]
        # Getting the upper and lower quartile of the curve: 25 and 75 percent
        lowerQuartile, upperQuartile = np.percentile(goundtruth_entry, [25, 75])
        # Isolating the middle entries
        iqr = upperQuartile - lowerQuartile

        binsInlier[binsTotalEntries] = (goundtruth_entry >= lowerQuartile - 1.5 * iqr) & (goundtruth_entry <= upperQuartile + 1.5*iqr)

    # Mask after getting the inliers
    predictions_cleaned, groundTruth_cleaned = predictions[binsInlier], groundTruth[binsInlier]
    # Isolating the values outside the standard deviation
    polyGraph = np.polyfit(predictions_cleaned, groundTruth_cleaned, deg=4)
    residual_enties = groundTruth_cleaned - np.polyval(polyGraph, predictions_cleaned)
    standardDevitions = np.abs(stats.zscore(residual_enties))
    # Having variable for sensitivity of the readings
    sensitivity = 2.75
    devitionsMask = standardDevitions < sensitivity

    return predictions_cleaned[devitionsMask], groundTruth_cleaned[devitionsMask]

# Function to find the specific type of relationship between relative and absolute accuracy
def model_hyperbolic(p, a, b, c):
    return a / (p + b) + c

def model_power(p, a, b, c):
    return a * np.power(np.clip(p, 1e-6, None), b) + c

def model_log(p, a, b, c):
    return a * np.log(np.clip(p + b, 1e-6, None)) + c

def model_exponential(p, a, b, c):
    return a * np.exp(b * p) + c

# Function to run all the individual models to compute the results and individual variables
def fit_all_models(p_clean, g_clean, n_sample=100_000):
    # Subsample — curve_fit doesn't need 3M points, 100k is plenty
    if len(p_clean) > n_sample:
        idx = np.random.choice(len(p_clean), n_sample, replace=False)
        p_fit = p_clean[idx]
        g_fit = g_clean[idx]
    else:
        p_fit, g_fit = p_clean, g_clean

    # Separate bounds per model — don't force hyperbolic bounds onto others
    models = {
        "hyperbolic": (model_hyperbolic, [-5.0, -0.85, 35.0],
                       ([-np.inf, -1.0 + 1e-6, -np.inf], [np.inf, -1e-6, np.inf])),
        "power":      (model_power,      [10.0, 0.5, 0.0],
                       ([-np.inf, -np.inf, -np.inf], [np.inf, np.inf, np.inf])),
        "log":        (model_log,        [-20.0, 1.0, 40.0],
                       ([-np.inf, -np.inf, -np.inf], [np.inf, np.inf, np.inf])),
        "exponential": (model_exponential, [2.2, 2.97, 5.6],
                       ([-np.inf, -np.inf, -np.inf], [np.inf, np.inf, np.inf])),
    }

    results = {}
    for name, (fn, p0, bounds) in models.items():
        try:
            popt, _ = curve_fit(fn, p_fit, g_fit, p0=p0, bounds=bounds, maxfev=50000)
            # Evaluate metrics on full cleaned set
            g_pred = fn(p_clean, *popt)
            ss_res = np.sum((g_clean - g_pred) ** 2)
            ss_tot = np.sum((g_clean - g_clean.mean()) ** 2)
            r2   = 1 - ss_res / ss_tot
            rmse = np.sqrt(np.mean((g_clean - g_pred) ** 2))
            results[name] = {"params": popt, "R2": r2, "RMSE": rmse}
            print(f"{name:12s}  R²={r2:.4f}  RMSE={rmse:.3f}m  params={np.round(popt, 4)}")
        except Exception as e:
            print(f"{name}: failed — {e}")
    return results

# Function to extract the best median curve possible
def extract_median_curve(p_clean, g_clean, n_bins=50):
    from scipy.interpolate import UnivariateSpline
    bins = np.linspace(p_clean.min(), p_clean.max(), n_bins + 1)
    bin_centers, medians, stds, counts = [], [], [], []
    for i in range(n_bins):
        m = (p_clean >= bins[i]) & (p_clean < bins[i+1])
        if m.sum() < 10:
            continue
        bin_centers.append((bins[i] + bins[i+1]) / 2)
        medians.append(np.median(g_clean[m]))
        stds.append(np.std(g_clean[m]))
        counts.append(m.sum())
    centers = np.array(bin_centers)
    meds    = np.array(medians)
    weights = np.sqrt(np.array(counts))
    spline  = UnivariateSpline(centers, meds, w=weights/weights.max(), s=len(centers)*0.5)
    return centers, meds, np.array(stds), spline

# Finding the best plot for the graph
def plot_fit_comparison(p_raw, g_raw, p_clean, g_clean, fit_results, spline):
    fig, axes = plt.subplots(1, 2, figsize=(16, 6))
    x_line = np.linspace(p_clean.min(), p_clean.max(), 300)
    fn_map = {"hyperbolic": model_hyperbolic, "power": model_power, "log": model_log, "exponential": model_exponential}
    colors = {'hyperbolic': 'red', 'power': 'green', 'log': 'orange', "exponential": 'purple'}

    for ax, (p, g, title) in zip(axes, [
        (p_raw,   g_raw,   "Raw (with noise)"),
        (p_clean, g_clean, "After preprocessing")
    ]):
        # Cap scatter at 5000 points — prevents hang
        n = min(5000, len(p))
        idx = np.random.choice(len(p), n, replace=False)
        ax.scatter(p[idx], g[idx], s=1, alpha=0.3, color='steelblue', label='data')

        for name, res in fit_results.items():
            y_line = fn_map[name](x_line, *res['params'])
            ax.plot(x_line, y_line, color=colors[name], lw=2,
                    label=f"{name}  R²={res['R2']:.3f}  RMSE={res['RMSE']:.2f}m")

        ax.plot(x_line, spline(x_line), 'k--', lw=2, label='median spline')
        ax.set_xlabel("Predicted Depth (relative)")
        ax.set_ylabel("Ground Truth Depth (m)")
        ax.set_title(title)
        ax.legend(fontsize=8)
        ax.set_ylim(0, 45)

    plt.tight_layout()
    os.makedirs("plots", exist_ok=True)
    plt.savefig("plots/fit_comparison.png", dpi=150)
    plt.show()

# Function to apply custom formula for each pixels in the image
def apply_custom_formula(pred_depth, formula_type=None, params=None):
    
    if formula_type is None or params is None:
        return pred_depth.copy()

    if formula_type == "linear":
        a = params["a"]
        b = params["b"]
        return a * pred_depth + b

    elif formula_type == "power":
        a = params["a"]
        b = params["b"]
        c = params.get("c", 0.0)
        return a * np.power(np.clip(pred_depth, 1e-6, None), b) + c

    elif formula_type == "exponential":
        a = params["a"]
        b = params["b"]
        c = params.get("c", 0.0)
        return a * np.exp(b * pred_depth) + c

    elif formula_type == "hyperbolic":
        a = params["a"]
        b = params["b"]
        c = params.get("c", 0.0)
        # Clip to avoid division by zero where pred_depth == -b
        return a / np.clip(pred_depth + b, 1e-6, None) + c

    elif formula_type == "log":
        a = params["a"]
        b = params["b"]
        c = params.get("c", 0.0)
        # Clip to avoid log(0) or log of negative
        return a * np.log(np.clip(pred_depth + b, 1e-6, None)) + c

    else:
        raise ValueError(f"Unknown formula_type: {formula_type}")

### Run Calibration

Applies the configured formula frame-by-frame across `pred_aligned`. Defaults to identity mapping when `FORMULA_TYPE = None`.

In [ ]:
FORMULA_TYPE = None
FORMULA_PARAMS = None

pred5 = pred_aligned[:5]
gt5   = gt_aligned[:5]

mask = np.isfinite(pred5) & np.isfinite(gt5) & (gt5 > 0) & (gt5 < 40)

pred40 = pred5[mask]
gt40 = gt5[mask]

# Running the calibration utility
# ── Run ──────────────────────────────────────────────────────────────
p_raw = pred40.flatten()
g_raw = gt40.flatten()

p_clean, g_clean = noiseHandler(pred40, gt40)
print(f"Points before cleaning: {len(pred40.flatten()):,}")
print(f"Points after  cleaning: {len(p_clean):,}")

fit_results = fit_all_models(p_clean, g_clean)
centers, medians, stds, spline = extract_median_curve(p_clean, g_clean)
plot_fit_comparison(p_raw, g_raw, p_clean, g_clean, fit_results, spline)

# RMSE evaluation on the full valid pixel set (0–40m)
eval_mask = np.isfinite(pred40) & np.isfinite(gt40) & (gt40 > 0) & (gt40 <= 40)
gt_eval   = gt40[eval_mask]
# --- Residuals ---
best_params = fit_results['exponential']['params']
pred_aligned_full = model_exponential(pred40[eval_mask], *best_params)
residuals = np.abs(pred_aligned_full - gt40[eval_mask])

# --- Masks ---
high_err_mask = residuals > 5.0
low_err_mask  = ~high_err_mask

num_bad  = high_err_mask.sum()
num_good = low_err_mask.sum()
total    = len(residuals)

# --- Find best model ---
fn_map = {
    "hyperbolic":  model_hyperbolic,
    "power":       model_power,
    "log":         model_log,
    "exponential": model_exponential,
}

best_name, best_rmse = None, float("inf")
pred40_eval = pred40[eval_mask] 

for name, res in fit_results.items():
    pred_transformed = fn_map[name](pred40_eval, *res["params"]) 
    rmse_full = np.sqrt(np.mean((pred_transformed - gt40[eval_mask]) ** 2))

    if rmse_full < best_rmse:
        best_rmse = rmse_full
        best_name = name

# --- Final output ---
print("\n=== FINAL SUMMARY ===")
print(f"Good pixels : {num_good:,}")
print(f"Bad pixels  : {num_bad:,}")
print(f"Total       : {total:,}")
print(f"Coverage    : {100 * num_good / total:.2f}%")

print(f"\nBest model  : {best_name}")
print(f"RMSE        : {best_rmse:.4f} m")

# Loading the remaning frames in a different stack
frame_pred = pred_aligned[5:]
FORMULA_TYPE = best_name
FORMULA_PARAMS = dict(zip(["a","b","c"], fit_results[best_name]["params"]))

pred_calibrated = np.stack(
    [apply_custom_formula(frame, FORMULA_TYPE, FORMULA_PARAMS) for frame in frame_pred],
    axis=0
)

print("pred_calibrated shape:", pred_calibrated.shape)
print("Currently using placeholder calibration (identity mapping).")

pred_calibrated shape: (197, 1280, 1920)
Currently using placeholder calibration (identity mapping).


## Save Final Outputs

Persists calibrated predictions and the aligned ground-truth tensor to disk so downstream metric scripts can load them without re-running inference.

In [20]:
CALIBRATED_PATH = OUTPUT_DIR / "depthanythingv2_calibrated_placeholder.npy"
PRED_STACK_PATH = OUTPUT_DIR / "depthanythingv2_depths.npy"
PRED_STEMS_PATH = OUTPUT_DIR / "depthanythingv2_stems.npy"
GT_ALIGNED_PATH = OUTPUT_DIR / "groundtruth_aligned.npy"

np.save(CALIBRATED_PATH, pred_calibrated)
np.save(GT_ALIGNED_PATH, gt_aligned)

print("Saved placeholder calibrated depth to:", CALIBRATED_PATH)
print("Saved aligned ground truth to:", GT_ALIGNED_PATH)

Saved placeholder calibrated depth to: C:\Users\19734\Desktop\DAVIS\winter quarter 2026\EEC 174AY\Senior Design\Eco-Cars-Depth-Estimation-2026\data\depth_anything_v2\depthanythingv2_calibrated_placeholder.npy
Saved aligned ground truth to: C:\Users\19734\Desktop\DAVIS\winter quarter 2026\EEC 174AY\Senior Design\Eco-Cars-Depth-Estimation-2026\data\depth_anything_v2\groundtruth_aligned.npy


## Finding RSME, AbsRel and δ1 accuracy

In [ ]:
# ── Cell 12: Out-of-sample validation ────────────────────────────────

pred_val = np.load("path/to/validation/pred.npy")
gt_val   = np.load("path/to/validation/gt.npy")

# Removing the first 5 frames, as they were used during calibration
pred_val = pred_val[5:]
gt_val = gt_val[5:]

# Same shape cleanup as Cell 2
if pred_val.ndim == 4 and pred_val.shape[-1] == 1:
    pred_val = pred_val[..., 0]
if gt_val.ndim == 4 and gt_val.shape[-1] == 1:
    gt_val = gt_val[..., 0]

# Valid pixel mask — same rules as training data
eval_mask_val = (
    np.isfinite(pred_val) &
    np.isfinite(gt_val)   &
    (gt_val <= 40.0)
)

gt_v   = gt_val[eval_mask_val]
pred_v = pred_val[eval_mask_val]

# Apply best formula — NO refitting
best_fn     = fn_map[best_name]
best_params = fit_results[best_name]['params']
pred_aligned = best_fn(pred_v, *best_params)

# ── Overall metrics ──
absrel_overall = np.mean(np.abs(pred_aligned - gt_v) / gt_v)
rmse_overall   = np.sqrt(np.mean((pred_aligned - gt_v) ** 2))
ratio          = np.maximum(pred_aligned / gt_v, gt_v / pred_aligned)
delta1_overall = np.mean(ratio < 1.25)

print(f"Overall  |  AbsRel: {absrel_overall:.4f}  RMSE: {rmse_overall:.4f}m  δ1: {delta1_overall:.4f}")
print()

# ── 8-bin stratified metrics ──
bin_edges = [0, 5, 10, 15, 20, 25, 30, 35, 40]

print(f"{'Bin':<10} {'N pts':<10} {'AbsRel':<10} {'RMSE (m)':<12} {'δ1 acc':<10}")
print("=" * 52)

for lo, hi in zip(bin_edges, bin_edges[1:]):
    bin_mask = (gt_v >= lo) & (gt_v < hi)
    if bin_mask.sum() < 10:
        print(f"{lo}-{hi}m     <10 pts — skip")
        continue

    gt_bin   = gt_v[bin_mask]
    pred_bin = pred_aligned[bin_mask]

    # Consistency filter — only pixels where pred and gt agree within 25%
    ratio_b        = np.maximum(pred_bin / gt_bin, gt_bin / pred_bin)
    consistent     = ratio_b < 1.25
    n_consistent   = consistent.sum()
    pct_consistent = 100 * n_consistent / len(gt_bin)

    gt_con   = gt_bin[consistent]
    pred_con = pred_bin[consistent]

    # Metrics on ALL points in bin
    absrel_all = np.mean(np.abs(pred_bin - gt_bin) / gt_bin)
    rmse_all   = np.sqrt(np.mean((pred_bin - gt_bin) ** 2))

    # Metrics on consistent points only
    absrel_con = np.mean(np.abs(pred_con - gt_con) / gt_con) if n_consistent > 0 else float('nan')
    rmse_con   = np.sqrt(np.mean((pred_con - gt_con) ** 2))  if n_consistent > 0 else float('nan')

    print(f"{lo}-{hi}m")
    print(f"  Consistent pts : {n_consistent:,} / {len(gt_bin):,}  ({pct_consistent:.1f}%)")
    print(f"  All pts        |  AbsRel: {absrel_all:.4f}  RMSE: {rmse_all:.4f}m")
    print(f"  Consistent only|  AbsRel: {absrel_con:.4f}  RMSE: {rmse_con:.4f}m")
    print()